Ziel: 

``
<toolshed>/repos/<owner>/<repository>/<tool_xml_id>/<version>
``


In [ ]:
import json
from bioblend.galaxy import GalaxyInstance
from bioblend.toolshed import ToolShedInstance
from typing import List, Dict, Any
from tqdm import tqdm

In [ ]:
def get_installed_repos(
        galaxy_url: str,
        galaxy_api_key: str = None,
):
    
    gi = GalaxyInstance(url=galaxy_url, key=galaxy_api_key)
    repos = gi.toolshed.get_repositories()

    installed_repos = [
        repo for repo in repos 
        if repo.get("status") == "Installed"
        and not repo.get("deleted", False)
        and not repo.get("uninstalled", False)
    ]

    return installed_repos
  

In [ ]:
def get_tools_for_repos(
        installed_repos: List[Dict[str, Any]]
):
    results: List[Dict[str, Any]] = []

    for repo in tqdm(installed_repos, desc="Lade Tools...", unit="repo"):
        owner = repo["owner"]
        name = repo["name"]
        revision = repo["changeset_revision"]

        ts_base = None
        domain = repo.get("tool_shed")
        ts_base = f"https://{domain}" if domain else None

        if not ts_base:
            results.append({
                "owner": owner,
                "name": name,
                "revision": revision,
                "tools": []
            })
            continue
        
        try:
            ts = ToolShedInstance(url=ts_base.rstrip("/"))
            ordered_revisions = ts.repositories.get_ordered_installable_revisions(name=name, owner=owner)
            latest_revision = ordered_revisions[-1]
            _, meta_data,*_ = ts.repositories.get_repository_revision_install_info(name=name, owner=owner, changeset_revision=latest_revision)

            tools_md = (meta_data or {}).get("tools") or (meta_data or {}).get("valid_tools") or []
            tools_out = []
            for t in tools_md:
                tools_out.append({
                    "tool_id": t.get("id"),
                    "version": t.get("version"),
                    "guid": t.get("guid") or f"{ts.base_url.rstrip('/')}/repos/{owner}/{name}/{t.get('id')}/{t.get('version')}"
                })

            results.append({
                "owner": owner,
                "name": name,
                "changeset_revision": latest_revision,
                "tool_shed_url": ts_base,
                "tools": tools_out
            })

        except Exception as e:
            results.append({
                "owner": owner, "name": name, "changeset_revision": revision,
                "tool_shed_url": ts_base, "tools": [],
                "error": str(e)
            })
    
    return results
        

        

    


In [ ]:
GALAXY_URL = "https://usegalaxy.eu"

installed_repos = get_installed_repos(
    galaxy_url=GALAXY_URL,
    galaxy_api_key=None)

# result = get_tools_for_repos(
#     installed_repos=installed_repos
#     )

print(json.dumps(installed_repos, indent=2))

# with open("tools_metadata.json", "w", encoding="utf-8") as f:
#     json.dump(result, f, ensure_ascii=False, indent=2)



In [15]:
with open("tools_metadata.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for repo in tqdm(data, desc="Load descriptions...", unit="repo"):
    owner = repo["owner"]
    name = repo["name"]
    ts_base = repo.get("tool_shed_url", "https://toolshed.g2.bx.psu.edu")
    revision = repo["changeset_revision"]
    
    try:
        ts = ToolShedInstance(url = ts_base.rstrip("/"))

        data_rev = ts.repositories.get_repository_revision_install_info(
            name=name,
            owner=owner,
            changeset_revision=revision)
        
        repo_info = data_rev[0]  
        repo_description = repo_info.get("description", "")
        repo_long_description = repo_info.get("long_description", "")

        meta_data = data_rev[1]
        tools_meta = meta_data.get("valid_tools", []) or meta_data.get("tools", [])

        for tool_entry in repo["tools"]:
            
            matching_tool = next(
                (t for t in tools_meta if t.get("id") == tool_entry.get("tool_id")), None
            )
            if matching_tool:
                tool_entry["name"] = matching_tool.get("name", "")
                tool_entry["description"] = matching_tool.get("description", repo_description)
                tool_entry["version"] = matching_tool.get("version", "")
                tool_entry["guid"] = matching_tool.get("guid", tool_entry.get("guid", ""))
            else:
                tool_entry["description"] = repo_description
            
            tool_entry["repo_description"] = repo_description
            tool_entry["repo_long_description"] = repo_long_description

    except Exception as e:
        repo["error"] = str(e)

with open("tools_metadata_with_full_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

Load descriptions...: 100%|██████████| 14099/14099 [3:18:51<00:00,  1.18repo/s]   
